# Kokoro TextToSpeech (TTS) Generator and TXT cleaner.

This delightful little tool leverages  [Kokoro TTS](https://huggingface.co/onnx-community/Kokoro-82M-v1.0-ONNX) and the [QWEN Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct) models to process .txt files within Google Colab, requiring absolutely no technical wizardry on your part.

Designed for exceptional simplicity, it transforms your reading material with just a few easy steps:

1. **Copy and Paste**: Gather your source text and paste it directly into Notepad, TextEdit, or your favorite text editor.

2. **Save**: Save the document as a standard .txt file. There is no need to fuss over formatting or tidy up the layout!

3. **Listen**: Run the file through this Jupyter Notebook, sit back, and let it spin your plain text into a beautiful, audiobook style masterpiece.

Consider it your personal, tireless narrator, ready to bring your documents to life at a moment's notice!

**Note**: The AI Text Cleaner tries its best to preserve your text, but it can be a tiny bit unpredictable! It might ***occasionally*** tweak wording or formatting, making each run slightly unique. Because of these little quirks, please don't use it for strict math or technical documents where exact punctuation is super important!

In [ ]:
# @title 🛠️ AI Text Processor & 🎙️ TTS (Text to Speech) Generator 🛠️
# @markdown ### Select your Task and Input Source below:
Task = "Both: Process Text then Generate TTS" # @param ["Text Processor Only", "TTS Generator Only", "Both: Process Text then Generate TTS"]
input_source = "Upload .txt File" # @param ["Text Box", "Upload .txt File"]
# @markdown <hr />

# @markdown ### ⚙️ General Settings:
save_to_google_drive = False # @param {type:"boolean"}
zip_password = "" # @param {type:"string"}
text = "Put your text here, this box is ignored if you are using a .txt file." # @param {type:"string"}
output_filename = "Processed_File" # @param {type:"string"}
# @markdown <hr />

# @markdown ### 🎙️ TTS Settings (Only used if generating audio):
voice = "af_nova" # @param ["af_nova", "af_heart", "bf_emma", "am_fenrir", "bm_daniel"] {allow-input: true}
# @markdown <hr />

# @markdown ### 🛠️ Advanced Settings
# @markdown <details><summary><b>👉 Click here to read about the Advanced Settings 👈</b></summary>
# @markdown <br />
# @markdown <b>system_prompt:</b> Sets the AI persona and core rules. Options:<br />
# @markdown * <i>Audio-Text Preparer:</i> Cleans text (removes bad breaks, footers, page numbers) without altering original words.<br />
# @markdown * <i>Storyteller / Epic Saga:</i> Expands input into rich creative prose.<br /><br />
# @markdown <b>chain_iterations:</b> Controls total length via iterative generation runs.<br />
# @markdown * <i>1 Iteration:</i> Best for standard formatting/cleaning tasks (~2048 max output tokens).<br />
# @markdown * <i>3-5 Iterations:</i> Best for long stories/epics (~6,000–10,000 output tokens).<br /><br />
# @markdown <b>model_choice:</b><br />
# @markdown * `Qwen/Qwen2.5-3B-Instruct`: Excellent precision for strict text cleanup.<br />
# @markdown * `dphn/Dolphin3.0-Qwen2.5-3b` & `dphn/Dolphin3.0-Llama3.2-3B`: Ideal for unconstrained creative writing.
# @markdown </details>

system_prompt = "You are an expert storyteller and creative writer. Your task is to write a highly detailed, immersive story based on the provided text. You must follow these rules strictly: 1. Pacing is crucial: take your time building the world, describing the sensory details of the environment, and establishing character motivations before advancing the plot. 3. Include rich, meaningful dialogue. 4. DO NOT summarize events; show them happening in real-time. 5. Output ONLY the story itself with no introductions or conclusions." # @param ["You are an expert audio-text preparer. Your task is to process this text so it reads smoothly for Text-to-Speech processing. 1. Remove random line breaks to reconstruct proper flowing paragraphs. 2. Fix broken hyphenations (e.g., 'para- graph' becomes 'paragraph'). 3. Normalize spacing by removing extra spaces or tabs. 4. Delete inline headers, footers, page numbers, and stray isolated numbers. 5. DO NOT rewrite, summarize, or change the author's original words. Output ONLY the processed text with no conversational filler.", "You are an expert storyteller and creative writer. Your task is to write a highly detailed, immersive story based on the provided text. You must follow these rules strictly: 1. Pacing is crucial: take your time building the world, describing the sensory details of the environment, and establishing character motivations before advancing the plot. 3. Include rich, meaningful dialogue. 4. DO NOT summarize events; show them happening in real-time. 5. Output ONLY the story itself with no introductions or conclusions.", "You are an expert storyteller and creative writer. Your task is to write an extraordinarily long, epic, and highly detailed saga based on the provided text. You must follow these rules strictly: 1. Take extreme amounts of time detailing the history of the world, inner thoughts of characters, and sensory details. 2. Include rich, extended dialogue. 3. Output ONLY the story itself with no introductions or conclusions."] {allow-input: true}
chain_iterations = 1 # @param {type:"slider", min:1, max:10, step:1}
model_choice = "Qwen/Qwen2.5-3B-Instruct" # @param ["Qwen/Qwen2.5-3B-Instruct", "dphn/Dolphin3.0-Qwen2.5-3b", "dphn/Dolphin3.0-Llama3.2-3B"] {allow-input: true}

import os
import sys
import subprocess
import shutil
import re
from IPython.display import Audio, display, clear_output
from google.colab import files

# --- BYPASS HUGGING FACE TOKEN POPUP ---
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TOKEN_WARNING"] = "1"

# --- MOUNT GOOGLE DRIVE FIRST ---
if save_to_google_drive:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print("📂 Mounting Google Drive, Please login using the popup window.")
        drive.mount('/content/drive')

# --- ENSURE 7-ZIP IS INSTALLED ---
if save_to_google_drive and zip_password.strip():
    try:
        subprocess.run(["7z"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except FileNotFoundError:
        print("📦 Installing 7-zip for encryption...")
        subprocess.run(["apt-get", "install", "-y", "p7zip-full"], stdout=subprocess.DEVNULL)

# --- 1. GET THE INPUT TEXT ---
raw_input = ""
base_name = output_filename

if input_source == "Upload .txt File":
    print("📂 Awaiting file upload... Please select your .txt file below.")
    uploaded = files.upload()
    if not uploaded:
        print("❌ No file uploaded. Execution stopped.")
        sys.exit()

    original_filename = list(uploaded.keys())[0]
    raw_input = uploaded[original_filename].decode('utf-8')
    print(f"✅ Loaded '{original_filename}' successfully.")

    base_name = os.path.splitext(original_filename)[0]
else:
    raw_input = text

if not raw_input.strip():
    print("⚠️ No text detected. Please paste text in the box or upload a file.")
    sys.exit()

current_text = raw_input

# --- 2. TEXT PROCESSOR LOGIC ---
if "Text Processor" in Task or "Both" in Task:
    print("\n" + "="*50)
    print("🖨️ STARTING AI TEXT PROCESSOR (CHAINING MODE)...")
    print("="*50)

    try:
        import transformers
    except ImportError:
        print("📦 Installing required libraries... (First run only)")
        subprocess.run(["pip", "install", "-q", "-U", "transformers", "accelerate", "torch"], check=True)

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if 'model' in globals() and globals().get('current_model_name') != model_choice:
        print("🧹 Unloading previous model to free up VRAM...")
        del globals()['model']
        del globals()['tokenizer']
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if 'model' not in globals() or 'tokenizer' not in globals():
        print(f"⏳ Loading {model_choice} into GPU... (Takes 2-3 minutes)")
        tokenizer = AutoTokenizer.from_pretrained(model_choice)
        model = AutoModelForCausalLM.from_pretrained(
            model_choice,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        current_model_name = model_choice
        print("✅ Model loaded successfully!")
    else:
        print(f"⚡ Model ({model_choice}) already in memory. Skipping setup.")

    def process_text_with_ai(raw_text):
        final_chained_output = ""

        # Adaptive temperature logic based on prompt intent
        is_story = "story" in system_prompt.lower() or "saga" in system_prompt.lower()
        gen_temperature = 0.75 if is_story else 0.1

        current_context = f"Please process the following text:\n\n{raw_text}"

        for step in range(chain_iterations):
            if chain_iterations > 1:
                print(f"   🔗 Generating chain link {step+1} of {chain_iterations}...")

            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": current_context}
            ]

            formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            model_inputs = tokenizer([formatted_prompt], return_tensors="pt").to(model.device)

            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=2048,           # Static token cap
                temperature=gen_temperature,
                repetition_penalty=1.15,       # Logit adjustment against repetition
                no_repeat_ngram_size=4,        # Prevents 4-word sequence loops
                do_sample=True,
            )

            new_tokens = generated_ids[0][len(model_inputs.input_ids[0]):]
            new_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

            final_chained_output += new_text + "\n\n"

            # Carry over the last ~300 words into the context window for the next link
            if step < chain_iterations - 1:
                last_few_words = " ".join(new_text.split()[-300:])
                current_context = f"Continue the story/text seamlessly from here:\n\n...{last_few_words}"

        return final_chained_output.strip()

    final_txt_filename = f"{base_name}_processed.txt"
    print(f"✨ Starting AI processing... Output will be saved to: {final_txt_filename}")

    chunk_size = 2500
    words = current_text.replace('\n', ' \n ').split(' ')
    chunks = []
    current_chunk = ""

    for word in words:
        if len(current_chunk) + len(word) + 1 < chunk_size:
            current_chunk += word + " "
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = word + " "
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    print(f"🧩 Document safely split into {len(chunks)} manageable chunks.")

    with open(final_txt_filename, "w", encoding="utf-8") as f:
        f.write("")

    for i, chunk in enumerate(chunks):
        print(f"⏳ Processing chunk {i+1} of {len(chunks)}...")
        processed_chunk = process_text_with_ai(chunk)

        with open(final_txt_filename, "a", encoding="utf-8") as f:
            f.write(processed_chunk + "\n\n")
            f.flush()
            os.fsync(f.fileno())

    print(f"🎉 Done! Processed text completely, saved to: {final_txt_filename}")

    with open(final_txt_filename, "r", encoding="utf-8") as f:
        current_text = f.read()

    if save_to_google_drive:
        if zip_password.strip():
            archive_name = f"{final_txt_filename}.7z"
            print(f"🔒 Encrypting {final_txt_filename}...")
            subprocess.run(["7z", "a", f"-p{zip_password}", "-mhe=on", archive_name, final_txt_filename], stdout=subprocess.DEVNULL)
            drive_path = f"/content/drive/MyDrive/{archive_name}"
            shutil.copy(archive_name, drive_path)
        else:
            drive_path = f"/content/drive/MyDrive/{final_txt_filename}"
            shutil.copy(final_txt_filename, drive_path)

        print(f"💾 Successfully saved text to Google Drive at: {drive_path}")

    try:
        if "Both" in Task:
            print(f"⚠️ Text processing finished. Colab will queue the text download to process at the very end of the cell.")
        files.download(final_txt_filename)
    except Exception as e:
        print(f"⚠️ Could not trigger automatic download. Find '{final_txt_filename}' on the left menu.")

# --- 3. TTS GENERATOR LOGIC ---
if "TTS Generator" in Task or "Both" in Task:
    print("\n" + "="*50)
    print("🎙️ STARTING KOKORO TTS GENERATOR...")
    print("="*50)

    try:
        import kokoro
        import soundfile as sf
    except ImportError:
        print("📦 Installing PyTorch Kokoro and dependencies...")
        subprocess.run(["pip", "install", "-q", "kokoro", "soundfile"], check=True)
        clear_output()
        import soundfile as sf

    import torch
    import numpy as np
    from kokoro import KPipeline

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"🚀 Computing Device: {device.upper()}")
    if device == 'cpu':
        print("⚠️ WARNING: GPU not detected. Generation will be slow. Go to Runtime > Change runtime type > select T4 GPU.")

    final_wav_filename = f"{base_name}_processed.wav" if "Both" in Task else f"{base_name}.wav"

    lang_code = voice[0]
    print(f"\nLoading Kokoro-82M model onto {device.upper()}...")
    pipeline = KPipeline(lang_code=lang_code, device=device)

    print(f"Generating speech for voice: {voice}...")

    text_chunks = [chunk.strip() for chunk in re.split(r'(?<=[.!?\n])\s+', current_text) if chunk.strip()]
    total_chunks = len(text_chunks)
    audio_chunks = []

    for i, chunk_text in enumerate(text_chunks):
        generator = pipeline(chunk_text, voice=voice, speed=1.0)
        for graphemes, phonemes, audio in generator:
            audio_chunks.append(audio)
        print(f"   -> Processed chunk {i+1} out of {total_chunks}...")

    if audio_chunks:
        print("Merging chunks and saving file...")
        final_audio = np.concatenate(audio_chunks)

        sf.write(final_wav_filename, final_audio, 24000)
        print(f"✅ Successfully created: {final_wav_filename}")

        if save_to_google_drive:
            if zip_password.strip():
                archive_name = f"{final_wav_filename}.7z"
                print(f"🔒 Encrypting {final_wav_filename} ...")
                subprocess.run(["7z", "a", f"-p{zip_password}", "-mhe=on", archive_name, final_wav_filename], stdout=subprocess.DEVNULL)
                drive_path = f"/content/drive/MyDrive/{archive_name}"
                shutil.copy(archive_name, drive_path)
            else:
                drive_path = f"/content/drive/MyDrive/{final_wav_filename}"
                shutil.copy(final_wav_filename, drive_path)

            print(f"💾 Successfully saved audio to Google Drive at: {drive_path}")

        display(Audio(final_wav_filename, autoplay=True))

        print("\n📥 Triggering Audio Download...")
        if "Both" in Task:
            print("⚠️ NOTE: Your browser may ask for permission to download multiple files at once. Please click 'Allow' in your URL bar if prompted.")

        try:
            files.download(final_wav_filename)
        except Exception as e:
            print(f"⚠️ Could not trigger automatic download. Find '{final_wav_filename}' on the left menu.")

    else:
        print("❌ Error: Audio generation failed.")